# Đề tài: Online Purchase Intention Prediction
## Môn: Học Máy (Machine Learning)

Dự án tập trung vào phân tích, tiền xử lý và đánh giá 5 mô hình học máy trên bộ dữ liệu **Online Shoppers Purchasing Intention**:
1. **Logistic Regression** (Hồi quy Logistic)
2. **K-Nearest Neighbors (KNN)**
3. **Decision Tree** (Cây quyết định)
4. **Random Forest** (Rừng ngẫu nhiên)
5. **Support Vector Machine (SVM)**

### 1. Import Thư Viện và Tải Dữ Liệu

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Đọc dữ liệu
df = pd.read_csv('data/online_shoppers_intention.csv')
print("Kích thước ban đầu:", df.shape)
df = df.drop_duplicates().reset_index(drop=True)
print("Sau khi loại bỏ trùng lặp:", df.shape)
df.head()

### 2. Phân Tích Dữ Liệu Khám Phá (EDA)
- Kiểm tra mất cân bằng nhãn mục tiêu (`Revenue`)
- Kiểm tra ma trận tương quan giữa các đặc trưng số

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='Revenue', ax=axes[0], palette='Blues_d', hue='Revenue', legend=False)
axes[0].set_title('Phân phối nhãn mục tiêu (Revenue)')

sns.boxplot(data=df, x='Revenue', y='PageValues', ax=axes[1], palette='Set2', hue='Revenue', legend=False)
axes[1].set_yscale('log')
axes[1].set_title('PageValues (Log Scale) theo Revenue')
plt.show()

print(df['Revenue'].value_counts(normalize=True))

### 3. Tiền Xử Lý Dữ Liệu (Preprocessing Pipeline)
- Tách đặc trưng số (numerical) và đặc trưng phân loại (categorical)
- Áp dụng `StandardScaler` cho đặc trưng số và `OneHotEncoder` cho đặc trưng phân loại.

In [ ]:
y = df['Revenue'].astype(int)
X = df.drop(columns=['Revenue'])

categorical_cols = ['Month', 'VisitorType', 'Weekend']
discrete_cat_cols = ['OperatingSystems', 'Browser', 'Region', 'TrafficType']
for col in discrete_cat_cols:
    X[col] = X[col].astype(str)

all_cat_cols = categorical_cols + discrete_cat_cols
numeric_cols = [c for c in X.columns if c not in all_cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), all_cat_cols)
    ]
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

### 4. Huấn Luyện & Đánh Giá 5 Mô Hình Học Máy

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights='distance'),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1),
    "SVM": SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42)
}

results = []
trained_pipes = {}

for name, clf in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['f1', 'roc_auc'], n_jobs=-1)
    
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'CV_F1': cv_scores['test_f1'].mean(),
        'CV_ROC_AUC': cv_scores['test_roc_auc'].mean(),
        'Test_Accuracy': accuracy_score(y_test, y_pred),
        'Test_Precision': precision_score(y_test, y_pred),
        'Test_Recall': recall_score(y_test, y_pred),
        'Test_F1': f1_score(y_test, y_pred),
        'Test_ROC_AUC': roc_auc_score(y_test, y_proba),
        'Test_PR_AUC': average_precision_score(y_test, y_proba)
    })

res_df = pd.DataFrame(results)
res_df

### 5. Biểu Đồ So Sánh Hiệu Năng (ROC, PR Curve & Feature Importance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for name, pipe in trained_pipes.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=axes[0])
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('Đường cong ROC trên Test Set')

for name, pipe in trained_pipes.items():
    PrecisionRecallDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=axes[1])
axes[1].set_title('Đường cong Precision-Recall trên Test Set')
plt.show()